# [Baseline] RandomForest — 풍력발전량 예측

기상 예보 데이터(LDAPS/GFS)로 KPX 발전그룹 3곳의 시간대별 **풍력발전량(kWh)** 을 예측하는 회귀 베이스라인입니다.

- **입력**: 예측 시각별 LDAPS/GFS 기상 예보 격자 평균값 + 시간·요일·계절성 캘린더 변수
- **출력**: `kpx_group_1/2/3` 세 발전그룹의 시간대별 예측 발전량(kWh)
- **모델**: 그룹별 `RandomForestRegressor` (Label 제공 기간이 그룹마다 다르므로 각각 따로 학습)

이 대회는 예측 결과 CSV(`baseline_submit.csv`)를 제출하는 방식이라, 데이터 로드부터 제출 파일 생성까지
이 노트북 하나로 처리합니다.


## 1. 라이브러리 불러오기

데이터 처리(pandas, numpy)와 회귀 모델 학습(scikit-learn)에 필요한 라이브러리를 불러옵니다.


In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer


## 2. 데이터 로드

발전량 Label(`train_labels.csv`), 제출 양식(`sample_submission.csv`), LDAPS/GFS 기상 예보 데이터를 불러오고,
날짜 컬럼을 datetime으로 변환합니다. `CAPACITY_KWH` 는 그룹별 설비 용량으로, 이후 예측값을 클리핑하는 데
사용합니다.


In [8]:
DATA_DIR = Path(r"C:\Users\user\iCloudDrive\riversun\동국대학교\3학년 1학기\대외활동\DACON\데이터\open")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}

train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

scada_vestas = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig")
scada_unison = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig")

print("train_labels:", train_labels.shape)
print("sample_submission:", sample_submission.shape)
print("ldaps_train:", ldaps_train.shape, "gfs_train:", gfs_train.shape)
print("ldaps_test:", ldaps_test.shape, "gfs_test:", gfs_test.shape)


train_labels: (26304, 4)
sample_submission: (8760, 5)
ldaps_train: (420864, 35) gfs_train: (236736, 40)
ldaps_test: (140160, 35) gfs_test: (78840, 40)


## 3. 데이터 전처리
P1. SCADA 글리치 제거

P2. Group 3 라벨 결측 처리

P3. NWP U/V → 풍속/풍향 변환

P4. Grid 선택

P5. NWP 풍속 Bias Correction

P6. Downtime 구간 플래깅

P7. Temporal Feature Engineering

P8. data_available_kst_dtm 활용 (leakage 방지)

P9. 10% 제외 규칙을 학습에 반영

P10. Train/Validation Split

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

# =====================================================================
# 1. [P10] 시간 컬럼 datetime 변환 및 통합 관리
# =====================================================================
for df in [train_labels, scada_vestas, scada_unison, ldaps_train, gfs_train, ldaps_test, gfs_test]:
    time_col = [col for col in df.columns if 'dtm' in col or 'time' in col][0]
    df[time_col] = pd.to_datetime(df[time_col])

# =====================================================================
# 2. [P8] 데이터 누수(Leakage) 방지: NWP 중복 제거 및 가용성 필터링
# =====================================================================
def clean_nwp(df):
    df = df[df['data_available_kst_dtm'] <= df['forecast_kst_dtm']]
    df = df.sort_values(by=['forecast_kst_dtm', 'data_available_kst_dtm'])
    df = df.drop_duplicates(subset=['forecast_kst_dtm', 'grid_id'], keep='last')
    return df

print("Cleaning NWP data...")
ldaps_train = clean_nwp(ldaps_train)
gfs_train = clean_nwp(gfs_train)
ldaps_test_clean = clean_nwp(ldaps_test)
gfs_test_clean = clean_nwp(gfs_test)


# =====================================================================
# 3. [P3, P4, P5] NWP 격자 전처리 및 물리 파생 변수 안전 생성
# =====================================================================
def process_nwp_features(df, prefix='ldaps'):
    # [💡 컴파일 방어 1] 시간 변수의 데이터 타입을 확실하게 datetime64로 강제 형변환
    df['forecast_kst_dtm'] = pd.to_datetime(df['forecast_kst_dtm'])
    df['data_available_kst_dtm'] = pd.to_datetime(df['data_available_kst_dtm'])
    
    df_idx = df[['forecast_kst_dtm', 'grid_id', 'data_available_kst_dtm']].copy()
    df_idx['forecast_horizon_h'] = (df_idx['forecast_kst_dtm'] - df_idx['data_available_kst_dtm']).dt.total_seconds() / 3600.0
    
    if prefix == 'ldaps':
        u_col = 'heightAboveGround_10_10u'
        v_col = 'heightAboveGround_10_10v'
        t_col = 'heightAboveGround_2_t'       
        p_col = 'surface_0_sp'                
        h_col = 'heightAboveGround_2_r'       
    else:  
        u_col = 'heightAboveGround_10_10u'
        v_col = 'heightAboveGround_10_10v'
        t_col = 'heightAboveGround_2_2t'      
        p_col = 'surface_0_sp'                
        h_col = 'heightAboveGround_2_2r'      
        
    rad_cols = [col for col in df.columns if any(r in col for r in ['dswrf', 'NDNSW', 'SWDIR', 'SWDIF'])]
    gust_cols = [c for c in df.columns if 'gust' in c.lower()]
    gust_col = gust_cols[0] if gust_cols else None
    
    target_base_cols = [u_col, v_col, t_col, p_col, h_col] + rad_cols
    if gust_col:
        target_base_cols.append(gust_col)
        
    imp_sub = SimpleImputer(strategy='median')
    df_sub = pd.DataFrame(imp_sub.fit_transform(df[target_base_cols]), columns=target_base_cols, index=df.index)
    
    for col in rad_cols:
        df_sub[col] = np.where(df_sub[col] < 0, 0, df_sub[col])

    # 대기 물리 변수 연산
    temp_c = df_sub[t_col] - 273.15  
    temp_k = df_sub[t_col]
    press_hpa = df_sub[p_col] / 100.0  
    
    e_s = 6.1078 * 10 ** ((7.5 * temp_c) / (temp_c + 237.3))
    e = (df_sub[h_col] / 100.0) * e_s
    p_dry = press_hpa - e
    df_sub['air_density'] = (p_dry / (287.058 * temp_k)) + ((e / (461.495 * temp_k)))
    
    alpha = ((17.27 * temp_c) / (237.7 + temp_c)) + np.log(0.5)
    dew_point = (237.7 * alpha) / (17.27 - alpha)
    e_s_dew = 6.112 * np.exp((17.67 * dew_point) / (dew_point + 243.5))
    e_a = e_s_dew * (df_sub[h_col] / 100.0)
    e_a_Pa = e_a * 100.0
    rho_d = (press_hpa * 100.0) / (287.058 * temp_k)
    rho_v = (e_a_Pa / (461.495 * temp_k))
    df_sub['absolute_humidity'] = (rho_v / (rho_d + rho_v)) * 1000.0

    # 풍속 및 오차 보정 계수 매핑
    df_sub['ws'] = np.sqrt(df_sub[u_col]**2 + df_sub[v_col]**2)
    bias_factor = 0.95 if prefix == 'ldaps' else 1.02
    df_sub['ws'] = df_sub['ws'] * bias_factor

    df_sub['ws_eff_zone'] = np.clip(df_sub['ws'] - 3.0, 0, 9.0)
    if gust_col:
        df_sub['gust_clean'] = np.clip(df_sub[gust_col], 0, 50.0)
        df_sub['ws_gust_ratio'] = df_sub['gust_clean'] / (df_sub['ws'] + 1e-5)
    else:
        df_sub['ws_gust_ratio'] = 1.2

    df_sub['wd'] = np.arctan2(-df_sub[u_col], -df_sub[v_col])
    df_sub['ws_sin'] = df_sub['ws'] * np.sin(df_sub['wd'])
    df_sub['ws_cos'] = df_sub['ws'] * np.cos(df_sub['wd'])
    df_sub['ws_squared'] = df_sub['ws'] ** 2
    df_sub['ws_cubed'] = df_sub['ws'] ** 3
    
    wd_deg = np.degrees(df_sub['wd']) % 360
    df_sub['wd_sector'] = (wd_deg / 45.0).astype(int)

    df_sub['forecast_kst_dtm'] = df_idx['forecast_kst_dtm']
    df_sub['forecast_horizon_h'] = df_idx['forecast_horizon_h']
    
    grouped = df_sub.groupby('forecast_kst_dtm').agg({
        'ws': ['mean', 'max', 'std'],
        'ws_squared': ['mean', 'max'],
        'ws_cubed': ['mean', 'max'],
        'ws_sin': 'mean',
        'ws_cos': 'mean',
        'air_density': 'mean',       
        'absolute_humidity': 'mean',
        'ws_eff_zone': 'mean',
        'ws_gust_ratio': 'mean',
        'forecast_horizon_h': 'mean',
        'wd_sector': 'median'
    })
    
    grouped.columns = [f"{prefix}_{c[0]}_{c[1]}" for c in grouped.columns]
    res_df = grouped.reset_index()
    return res_df

print("Extracting NWP features...")
ldaps_features = process_nwp_features(ldaps_train, prefix='ldaps')
gfs_features = process_nwp_features(gfs_train, prefix='gfs')
ldaps_test_features = process_nwp_features(ldaps_test_clean, prefix='ldaps')
gfs_test_features = process_nwp_features(gfs_test_clean, prefix='gfs')


# =====================================================================
# 4. [P1 & P6] 터빈 SCADA 글리치 제거 및 다운타임 비율 산출
# =====================================================================
print("Processing SCADA glitches for all Turbines...")
scada_vestas['hourly_kst'] = scada_vestas['kst_dtm'].dt.floor('h')
scada_unison['hourly_kst'] = scada_unison['kst_dtm'].dt.floor('h')

# Vestas
vestas_cap = 780
vestas_all_wtgs = [f'wtg{i:02d}' for i in range(1, 13)]
scada_vestas['total_downtime_count'] = 0
scada_vestas['active_turbine_count'] = 0

for wtg in vestas_all_wtgs:
    power_col = f'vestas_{wtg}_power_kw10m'
    ws_col = f'vestas_{wtg}_ws'
    scada_vestas[power_col] = np.where(scada_vestas[power_col] > vestas_cap * 1.3, np.nan, scada_vestas[power_col])
    scada_vestas[power_col] = np.where(scada_vestas[power_col] < 0, 0, scada_vestas[power_col])
    is_down = np.where((scada_vestas[ws_col] >= 5.0) & (scada_vestas[power_col] <= 0), 1, 0)
    valid_mask = scada_vestas[power_col].notna() & scada_vestas[ws_col].notna()
    scada_vestas['total_downtime_count'] += np.where(valid_mask, is_down, 0)
    scada_vestas['active_turbine_count'] += np.where(valid_mask, 1, 0)

scada_vestas['scada_downtime_ratio_10m'] = np.where(scada_vestas['active_turbine_count'] > 0, scada_vestas['total_downtime_count'] / scada_vestas['active_turbine_count'], 0)
vestas_hourly_down = scada_vestas.groupby('hourly_kst')['scada_downtime_ratio_10m'].mean().reset_index()
vestas_hourly_down.columns = ['kst_dtm', 'vestas_downtime_ratio']

# Unison
unison_cap = 750
unison_all_wtgs = [f'wtg{i:02d}' for i in range(1, 6)]
scada_unison['total_downtime_count'] = 0
scada_unison['active_turbine_count'] = 0

for wtg in unison_all_wtgs:
    power_col = f'unison_{wtg}_power_kw10m'
    ws_col = f'unison_{wtg}_ws'
    wd_col = f'unison_{wtg}_wd'
    scada_unison[power_col] = np.where(scada_unison[power_col] > unison_cap * 1.3, np.nan, scada_unison[power_col])
    scada_unison[power_col] = np.where(scada_unison[power_col] < 0, 0, scada_unison[power_col])
    if wd_col in scada_unison.columns:
        scada_unison[wd_col] = np.where(scada_unison[wd_col] < 0, scada_unison[wd_col] + 360, scada_unison[wd_col])
    is_down = np.where((scada_unison[ws_col] >= 5.0) & (scada_unison[power_col] <= 0), 1, 0)
    valid_mask = scada_unison[power_col].notna() & scada_unison[ws_col].notna()
    scada_unison['total_downtime_count'] += np.where(valid_mask, is_down, 0)
    scada_unison['active_turbine_count'] += np.where(valid_mask, 1, 0)

scada_unison['scada_downtime_ratio_10m'] = np.where(scada_unison['active_turbine_count'] > 0, scada_unison['total_downtime_count'] / scada_unison['active_turbine_count'], 0)
unison_hourly_down = scada_unison.groupby('hourly_kst')['scada_downtime_ratio_10m'].mean().reset_index()
unison_hourly_down.columns = ['kst_dtm', 'unison_downtime_ratio']


# =====================================================================
# 5. [P7] Temporal Feature Engineering
# =====================================================================
def add_temporal_features(df, time_col):
    df['hour'] = df[time_col].dt.hour
    df['month'] = df[time_col].dt.month
    df['day_of_year'] = df[time_col].dt.dayofyear
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    return df


# =====================================================================
# 6. Train / Validation 및 Test 최종 통합 프레임 조립
# =====================================================================
print("Merging all features...")
final_df = train_labels.copy()
final_df = pd.merge(final_df, ldaps_features, left_on='kst_dtm', right_on='forecast_kst_dtm', how='left')
final_df = pd.merge(final_df, gfs_features, left_on='kst_dtm', right_on='forecast_kst_dtm', how='left')
final_df = add_temporal_features(final_df, 'kst_dtm')

final_df = pd.merge(final_df, vestas_hourly_down, on='kst_dtm', how='left').fillna({'vestas_downtime_ratio': 0})
final_df = pd.merge(final_df, unison_hourly_down, on='kst_dtm', how='left').fillna({'unison_downtime_ratio': 0})
final_df = final_df.sort_values('kst_dtm').reset_index(drop=True)

# 🎯 [💡 컴파일 보정 2] 진짜 생성된 컬럼 구조로 싱크 매칭 수정 완료
for prefix in ['ldaps', 'gfs']:
    final_df[f'{prefix}_ws_mean_lag1'] = final_df[f'{prefix}_ws_mean'].shift(1)
    final_df[f'{prefix}_ws_mean_lead1'] = final_df[f'{prefix}_ws_mean'].shift(-1)
    final_df[f'{prefix}_ws_rolling_3h'] = final_df[f'{prefix}_ws_mean'].rolling(window=3, min_periods=1).mean()

time_lag_cols = [f'{p}_ws_mean_lag1' for p in ['ldaps', 'gfs']] + [f'{p}_ws_mean_lead1' for p in ['ldaps', 'gfs']] + [f'{p}_ws_rolling_3h' for p in ['ldaps', 'gfs']]
final_df[time_lag_cols] = final_df[time_lag_cols].bfill().ffill()

historical_downtime = final_df.groupby(['month', 'hour'])[['vestas_downtime_ratio', 'unison_downtime_ratio']].mean().reset_index()

test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
test_df = pd.merge(test_df, ldaps_test_features, left_on='forecast_kst_dtm', right_on='forecast_kst_dtm', how='left')
test_df = pd.merge(test_df, gfs_test_features, left_on='forecast_kst_dtm', right_on='forecast_kst_dtm', how='left')
test_df = add_temporal_features(test_df, 'forecast_kst_dtm')

test_df = test_df.sort_values('forecast_kst_dtm').reset_index(drop=True)
for prefix in ['ldaps', 'gfs']:
    test_df[f'{prefix}_ws_mean_lag1'] = test_df[f'{prefix}_ws_mean'].shift(1)
    test_df[f'{prefix}_ws_mean_lead1'] = test_df[f'{prefix}_ws_mean'].shift(-1)
    test_df[f'{prefix}_ws_rolling_3h'] = test_df[f'{prefix}_ws_mean'].rolling(window=3, min_periods=1).mean()
test_df[time_lag_cols] = test_df[time_lag_cols].bfill().ffill()

test_df = pd.merge(test_df, historical_downtime, on=['month', 'hour'], how='left').fillna(0)


# =====================================================================
# 7. [P2] Group 3 라벨 결측 처리 (Linear Regression Imputation)
# =====================================================================
valid_g3 = final_df[final_df['kpx_group_3'].notnull() & final_df['kpx_group_1'].notnull() & final_df['kpx_group_2'].notnull()]
missing_g3 = final_df[final_df['kpx_group_3'].isnull() & final_df['kpx_group_1'].notnull() & final_df['kpx_group_2'].notnull()]

if len(missing_g3) > 0:
    print("Imputing Group 3 missing labels...")
    lr = LinearRegression()
    lr.fit(valid_g3[['kpx_group_1', 'kpx_group_2']], valid_g3['kpx_group_3'])
    pred_g3 = lr.predict(missing_g3[['kpx_group_1', 'kpx_group_2']])
    final_df.loc[missing_g3.index, 'kpx_group_3'] = pred_g3


# =====================================================================
# 8. Train/Validation Split 및 모델 인풋 바인딩 완공
# =====================================================================
train_set = final_df[final_df['kst_dtm'].dt.year.isin([2022, 2023])].copy()
valid_set = final_df[final_df['kst_dtm'].dt.year == 2024].copy()

drop_features = ['kst_dtm', 'forecast_kst_dtm', 'data_available_kst_dtm', 'grid_id']

X_train = train_set.drop(columns=TARGET_COLS + [col for col in drop_features if col in train_set.columns], errors='ignore')
y_train = train_set[TARGET_COLS]

X_val = valid_set.drop(columns=TARGET_COLS + [col for col in drop_features if col in valid_set.columns], errors='ignore')
y_val = valid_set[TARGET_COLS]

X_test = test_df.drop(columns=[col for col in drop_features if col in test_df.columns], errors='ignore')
X_test = X_test.reindex(columns=X_train.columns)

print(f"🥇 [싱크 보정 완료] 전처리 완공! Train: {X_train.shape}, Valid: {X_val.shape}, Test: {X_test.shape}")

Cleaning NWP data...
Extracting NWP features...
Processing SCADA glitches for all Turbines...
Merging all features...
Imputing Group 3 missing labels...
🥇 [싱크 보정 완료] 전처리 완공! Train: (17519, 45), Valid: (8784, 45), Test: (8760, 45)


## 4. Feature 생성

LDAPS/GFS는 하나의 예측 시각에 여러 격자가 존재하므로, `forecast_kst_dtm`별로 수치형 기상변수의 평균값을 계산합니다.

추가로 월, 일, 시간, 요일, 주말 여부, 시간/월의 주기성을 나타내는 sin-cos feature를 생성합니다.


In [10]:
# ==========================================
# 4번 셀 전체 교체 (데이터 인젝션 및 구조 완벽 고정 버전)
# ==========================================

# 1. TEST 데이터도 똑같이 NWP 중복 제거 및 가용성 필터링 (P8)
print("Cleaning TEST NWP data...")
ldaps_test_clean = clean_nwp(ldaps_test)
gfs_test_clean = clean_nwp(gfs_test)

# 2. TEST 데이터도 똑같이 U/V -> 풍속/풍향 물리 변환 및 격자 집계 (P3, P4)
print("Extracting TEST NWP features...")
ldaps_test_features = process_nwp_features(ldaps_test_clean, prefix='ldaps')
gfs_test_features = process_nwp_features(gfs_test_clean, prefix='gfs')

# 3. TEST 데이터 최종 조립 (test_df 생성)
print("Merging TEST features...")
test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
test_df = pd.merge(test_df, ldaps_test_features, left_on='forecast_kst_dtm', right_on='forecast_kst_dtm', how='left')
test_df = pd.merge(test_df, gfs_test_features, left_on='forecast_kst_dtm', right_on='forecast_kst_dtm', how='left')

# 4. TEST 데이터에도 동일한 시간 파생 변수 인코딩 적용 (P7)
test_df = add_temporal_features(test_df, 'forecast_kst_dtm')

# TEST 데이터셋에도 동일하게 시간 흐름 피처 반영
test_df = test_df.sort_values('forecast_kst_dtm').reset_index(drop=True)
test_df['ldaps_ws_mean_lag1'] = test_df['ldaps_ws_mean'].shift(1)
test_df['ldaps_ws_mean_lead1'] = test_df['ldaps_ws_mean'].shift(-1)
test_df['ldaps_ws_rolling_3h'] = test_df['ldaps_ws_mean'].rolling(window=3, min_periods=1).mean()

test_df['gfs_ws_mean_lag1'] = test_df['gfs_ws_mean'].shift(1)
test_df['gfs_ws_mean_lead1'] = test_df['gfs_ws_mean'].shift(-1)
test_df['gfs_ws_rolling_3h'] = test_df['gfs_ws_mean'].rolling(window=3, min_periods=1).mean()

test_df[time_lag_cols] = test_df[time_lag_cols].bfill().ffill()

# TEST 세트에는 실측 가동률을 알 수 없으므로 0으로 채워 구조적 결합 유지
test_df['vestas_downtime_ratio'] = 0.0
test_df['unison_downtime_ratio'] = 0.0  

# 🎯 [핵심 동기화] 3번 셀에서 부활한 final_df를 안전하게 상속받아 이 자리에서 즉시 새로 쪼갭니다.
# 이렇게 해야 이전 메모리 잔류로 인한 싱크 어긋남 현상이 완벽하게 차단됩니다.
current_train_set = final_df[final_df['kst_dtm'].dt.year.isin([2022, 2023])].copy()
current_valid_set = final_df[final_df['kst_dtm'].dt.year == 2024].copy()

# 🎯 진짜 불필요한 인덱스성 변수만 정밀 드롭
drop_features = [
    'kst_dtm', 'forecast_kst_dtm', 'data_available_kst_dtm', 'grid_id'
]

# 분리 연산 수행
X_train = current_train_set.drop(columns=TARGET_COLS + [col for col in drop_features if col in current_train_set.columns], errors='ignore')
y_train = current_train_set[TARGET_COLS]

X_val = current_valid_set.drop(columns=TARGET_COLS + [col for col in drop_features if col in current_valid_set.columns], errors='ignore')
y_val = current_valid_set[TARGET_COLS]

X_test = test_df.drop(columns=[col for col in drop_features if col in test_df.columns], errors='ignore')
X_test = X_test.reindex(columns=X_train.columns)

print("\n--- 파이프라인 최종 조립 완료 ---")
print("X_train (학습 피처):", X_train.shape)
print("X_val   (검증 피처):", X_val.shape)
print("X_test  (평가 피처):", X_test.shape)

Cleaning TEST NWP data...
Extracting TEST NWP features...
Merging TEST features...

--- 파이프라인 최종 조립 완료 ---
X_train (학습 피처): (17519, 45)
X_val   (검증 피처): (8784, 45)
X_test  (평가 피처): (8760, 45)


In [11]:
# # === 4.5번 셀: 변수명 완벽 격리 + 시드 고정 버전 ===
# import optuna
# from optuna.samplers import TPESampler
# import lightgbm as lgb
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.impute import SimpleImputer

# print("🔥 [ALL TARGETS] 대회 산식(FICR) 최대화 변수 격리 튜닝 시작...")

# imputer = SimpleImputer(strategy="median")
# X_train_numeric = X_train.select_dtypes(include=[np.number])
# X_train_imp_local = pd.DataFrame(imputer.fit_transform(X_train_numeric), columns=X_train_numeric.columns)

# best_params_dict = {}

# def calculate_custom_ficr(y_true, y_pred, capacity):
#     eval_mask = y_true >= (capacity * 0.1)
#     if eval_mask.sum() == 0:
#         return 0
#     y_true_eval = y_true[eval_mask]
#     y_pred_eval = y_pred[eval_mask]
    
#     absolute_errors = np.abs(y_pred_eval - y_true_eval)
#     hourly_nmae = absolute_errors / capacity
    
#     revenue = np.where(hourly_nmae <= 0.06, 4.0, 
#                        np.where(hourly_nmae <= 0.08, 3.0, 0.0))
#     max_revenue = len(y_true_eval) * 4.0
#     return revenue.sum() / max_revenue

# for target_col in TARGET_COLS:
#     print(f"\n🎯 [{target_col}] 파라미터 사냥 중...")
    
#     train_mask = train_set[target_col].notna()
#     X_opt = X_train_imp_local.loc[train_mask]
#     y_opt = train_set.loc[train_mask, target_col].values
    
#     # 🎯 [핵심] X_val 변수명이 오염되지 않도록 뒤에 _opt를 붙여 철저히 분리합니다.
#     X_tr_opt, X_val_opt, y_tr_opt, y_val_opt = train_test_split(X_opt, y_opt, test_size=0.2, random_state=42)
#     target_capacity = CAPACITY_KWH[target_col]
    
#     def objective(trial):
#         params = {
#             'n_estimators': trial.suggest_int('n_estimators', 400, 1200, step=50),
#             'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.06, log=True),
#             'max_depth': trial.suggest_int('max_depth', 6, 12),
#             'num_leaves': trial.suggest_int('num_leaves', 31, 127),
#             'min_child_samples': trial.suggest_int('min_child_samples', 15, 35),
#             'subsample': trial.suggest_float('subsample', 0.65, 0.95),
#             'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 0.95),
#             'random_state': 42,
#             'n_jobs': -1,
#             'verbose': -1
#         }
        
#         model = lgb.LGBMRegressor(**params)
#         model.fit(X_tr_opt, y_tr_opt)
        
#         preds = model.predict(X_val_opt)
#         preds = np.clip(preds, 0, target_capacity)
        
#         ficr_score = calculate_custom_ficr(y_val_opt, preds, target_capacity)
#         return 1.0 - ficr_score

#     fixed_sampler = TPESampler(seed=42)
#     study = optuna.create_study(direction='minimize', sampler=fixed_sampler)
#     study.optimize(objective, n_trials=20)
    
#     best_params_dict[target_col] = study.best_trial.params
#     print(f"▶ [{target_col}] 최고 가상 FICR: {1.0 - study.best_value:.5f}")

# print("\n🏆 모든 타겟 격리 사냥 완료!")

In [12]:
# # 🎯 [완벽 보정] 진짜 부활한 기상 변수들이 들어있는지 정확하게 저격 검증
# check_cols = [col for col in X_train.columns if any(w in col for w in ['ws', 'density', 'humidity', 'pressure'])]

# if len(check_cols) > 0:
#     print("✅ 성공: 부활한 기온/기압/대기밀도 변수들이 모델 입력(X_train)에 100% 포함되어 있습니다!")
#     print(f"▶ X_train의 최종 피처 개수: {X_train.shape[1]}개")
#     print(f"▶ 포함된 소중한 기상 피처들: {check_cols[:5]}")
# else:
#     print("❌ 경고: 기상 데이터가 정상적으로 수혈되지 않았습니다.")

## 5. 모델 학습

KPX 그룹별로 Label 제공 기간이 다르므로, 각 그룹의 Label이 존재하는 행만 사용해 RandomForest 모델을 따로 학습합니다.

RandomForest는 여러 결정트리를 앙상블하는 배깅(Bagging) 모델로, 각 트리를 부트스트랩 샘플과 랜덤 피처
subset으로 학습한 뒤 예측을 평균 냅니다.


In [13]:
# === [산식 극대화 보정] LightGBM 단일 모델 5-Fold 파이프라인 ===
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def calculate_fold_ficr(y_true, y_pred, capacity):
    valid = y_true >= (capacity * 0.10)
    if np.sum(valid) == 0: return 0.0, 0.0
    actual = y_true[valid]
    forecast = y_pred[valid]
    
    error_rate = np.abs(forecast - actual) / capacity
    nmae_score = 1.0 - np.mean(error_rate)
    
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    ficr_score = np.sum(actual * unit_price) / np.sum(actual * 4.0)
    return nmae_score, ficr_score

# 1. 수치형 데이터 임퓨팅 및 컬럼 순서 강제 동기화
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
X_test_numeric = X_test_numeric.reindex(columns=X_train_numeric.columns)

imputer = SimpleImputer(strategy="median")
X_train_trans = imputer.fit_transform(X_train_numeric)
passed_features = imputer.get_feature_names_out(X_train_numeric.columns)

X_train_imp = pd.DataFrame(X_train_trans, columns=passed_features)
X_test_imp = pd.DataFrame(imputer.transform(X_test_numeric), columns=passed_features)

final_test_predictions = pd.DataFrame(index=sample_submission.index)
oof_predictions = pd.DataFrame(0.0, index=train_set.index, columns=TARGET_COLS)

models_dict = {target: {'lgb': []} for target in TARGET_COLS}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("⚡ [산식 타겟팅 최적화] LightGBM 단일 모델 5-Fold Cross Validation 시작...")

for target in TARGET_COLS:
    print(f"\n==================== 🎯 TARGET: {target} ====================")
    
    train_mask = train_set[target].notna()
    X_target = X_train_imp.loc[train_mask].reset_index(drop=True)
    y_target = train_set.loc[train_mask, target].reset_index(drop=True)
    target_indices = train_set.loc[train_mask].index
    
    target_capacity = CAPACITY_KWH[target]
    test_pred_folds = np.zeros(len(X_test_imp))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_target, y_target)):
        X_tr, y_tr = X_target.iloc[train_idx], y_target.iloc[train_idx]
        X_va, y_va = X_target.iloc[val_idx], y_target.iloc[val_idx]
        
        # 🎯 [산식 보정 핵심 1] 산식 평가 대상인 설비용량 10% 이상 구간에 가중치 대폭 집중
        # 10% 미만 구간은 예측 오차가 패널티로 작용하지 않으므로, 모델이 고발전 구간(6~8% 오차 이내 진입)에만 목숨 걸도록 유도합니다.
        threshold = target_capacity * 0.10
        sample_weights = np.where(y_tr < threshold, 0.01, 1.0) 
        
        # 🎯 [산식 보정 핵심 2] 손실 함수(Objective)를 MAE(regression_l1)로 변경
        # NMAE 정형 산식은 절대오차(L1) 기반이므로, MSE(L2)보다 MAE 손실 함수가 6%~8% 경계선 타격에 훨씬 유리합니다.
        model_lgb = lgb.LGBMRegressor(
            objective='regression_l1',
            n_estimators=800, learning_rate=0.05, max_depth=7,
            num_leaves=63, min_child_samples=20, subsample=0.8,
            colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1
        )
        model_lgb.fit(X_tr, y_tr, sample_weight=sample_weights)
        models_dict[target]['lgb'].append(model_lgb)
        
        # 예측 및 클리핑
        val_preds = np.clip(model_lgb.predict(X_va), 0, target_capacity)
        oof_predictions.loc[target_indices[val_idx], target] = val_preds
        
        # Test 예측 누적
        test_pred_folds += np.clip(model_lgb.predict(X_test_imp), 0, target_capacity) / kf.n_splits
        
        f_nmae, f_ficr = calculate_fold_ficr(y_va.to_numpy(), val_preds, target_capacity)
        print(f"▶ Fold {fold+1} Done! | 1-NMAE: {f_nmae:.4f}, FICR: {f_ficr:.4f}")
        
    final_test_predictions[target] = test_pred_folds

print("\n🏆 산식 맞춤형 LightGBM 모델 훈련 완료! 7번 셀을 실행해 변화된 점수를 확인하세요.")

⚡ [산식 타겟팅 최적화] LightGBM 단일 모델 5-Fold Cross Validation 시작...

==================== 🎯 TARGET: kpx_group_1 ====================
▶ Fold 1 Done! | 1-NMAE: 0.8934, FICR: 0.4627
▶ Fold 2 Done! | 1-NMAE: 0.8928, FICR: 0.4463
▶ Fold 3 Done! | 1-NMAE: 0.8899, FICR: 0.4574
▶ Fold 4 Done! | 1-NMAE: 0.8930, FICR: 0.4541
▶ Fold 5 Done! | 1-NMAE: 0.8944, FICR: 0.4584

==================== 🎯 TARGET: kpx_group_2 ====================
▶ Fold 1 Done! | 1-NMAE: 0.8889, FICR: 0.4709
▶ Fold 2 Done! | 1-NMAE: 0.8889, FICR: 0.4615
▶ Fold 3 Done! | 1-NMAE: 0.8887, FICR: 0.4753
▶ Fold 4 Done! | 1-NMAE: 0.8903, FICR: 0.4724
▶ Fold 5 Done! | 1-NMAE: 0.8928, FICR: 0.4909

==================== 🎯 TARGET: kpx_group_3 ====================
▶ Fold 1 Done! | 1-NMAE: 0.8936, FICR: 0.4604
▶ Fold 2 Done! | 1-NMAE: 0.8896, FICR: 0.4278
▶ Fold 3 Done! | 1-NMAE: 0.8913, FICR: 0.4565
▶ Fold 4 Done! | 1-NMAE: 0.8944, FICR: 0.4613
▶ Fold 5 Done! | 1-NMAE: 0.8994, FICR: 0.4704

🏆 산식 맞춤형 LightGBM 모델 훈련 완료! 7번 셀을 실행해 변화된 점수를 확인하세요.


## 6. 테스트 데이터 예측 생성

학습한 그룹별 모델로 평가 기간 전체의 발전량을 예측합니다. 예측값은 0 이상, 설비 용량 이하로 클리핑하여
물리적으로 불가능한 값을 방지합니다.


In [14]:
# === 6번 셀: 테스트 데이터 예측 생성 (변수명 보정 완료) ===
submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()

for target in TARGET_COLS:
    # 🎯 [핵심 보정] 5-Fold 교차 검증의 평균값인 final_test_predictions를 매칭합니다.
    submission[target] = final_test_predictions[target]

submission["forecast_kst_dtm"] = pd.to_datetime(submission["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")

print(submission.head())
print(submission.shape)

     forecast_id     forecast_kst_dtm   kpx_group_1   kpx_group_2  \
0  forecast_0001  2025-01-01 01:00:00  18747.715140  19933.835294   
1  forecast_0002  2025-01-01 02:00:00  19072.583522  19857.941476   
2  forecast_0003  2025-01-01 03:00:00  18225.918771  19647.235423   
3  forecast_0004  2025-01-01 04:00:00  18549.386524  19895.294044   
4  forecast_0005  2025-01-01 05:00:00  18142.193277  19441.278232   

    kpx_group_3  
0  16878.131273  
1  16875.597495  
2  16740.640898  
3  16654.098482  
4  16503.230190  
(8760, 5)


## 7. 평가 산식

In [15]:
# ====================================================
# 7번 셀 전체 교체 (대회 산식 함수 수혈 및 동기화 버전)
# ====================================================

# 🎯 데이콘 공식 메트릭 함수 정의 (7번 셀 메모리에 직접 주입)
def metric(answer_df, pred_df):
    group_nmae = []
    group_ficr = []

    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float)
        forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]

        valid = actual >= capacity * 0.10

        actual = actual[valid]
        forecast = forecast[valid]

        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))

        unit_price = np.select(
            [error_rate <= 0.06, error_rate <= 0.08],
            [4.0, 3.0],
            default=0.0,
        )

        earned_settlement = np.sum(actual * unit_price)
        max_settlement = np.sum(actual * 4.0)

        group_ficr.append(earned_settlement / max_settlement)

    one_minus_nmae = 1 - np.mean(group_nmae)
    ficr = np.mean(group_ficr)

    total_score = 0.5 * one_minus_nmae + 0.5 * ficr

    return total_score, one_minus_nmae, ficr


# === [보정용 7번 셀] LightGBM 단일 모델 로컬 스코어 리포트 ===
print("Calculating Local Validation Score with LightGBM Single Model...")

answer_df = current_valid_set[TARGET_COLS].copy()
X_val_numeric = X_val.select_dtypes(include=[np.number]).reindex(columns=X_train_numeric.columns)
X_val_trans = imputer.transform(X_val_numeric)
X_val_imp = pd.DataFrame(X_val_trans, columns=passed_features)

pred_df = pd.DataFrame(index=current_valid_set.index)

for target in TARGET_COLS:
    target_capacity = CAPACITY_KWH[target]
    val_preds_accum = np.zeros(len(X_val_imp))
    
    num_folds = len(models_dict[target]['lgb'])
    for i in range(num_folds):
        val_preds_accum += models_dict[target]['lgb'][i].predict(X_val_imp)
    val_preds_accum /= num_folds
        
    pred_df[target] = np.clip(val_preds_accum, 0, target_capacity)

# 최종 Hybrid 로컬 스코어 계산
total_score, one_minus_nmae, ficr = metric(answer_df, pred_df)

print("\n==========================================")
print("🎯 LIGHTGBM SINGLE LOCAL VALIDATION REPORT (2024)")
print("==========================================")
print(f"▶ 최종 점수 (Total Score) : {total_score:.5f}")
print(f"▶ 1-NMAE                 : {one_minus_nmae:.5f}")
print(f"▶ 정산금획득률 (FICR)     : {ficr:.5f}")
print("==========================================")

Calculating Local Validation Score with LightGBM Single Model...

🎯 LIGHTGBM SINGLE LOCAL VALIDATION REPORT (2024)
▶ 최종 점수 (Total Score) : 0.61636
▶ 1-NMAE                 : 0.87459
▶ 정산금획득률 (FICR)     : 0.35813


## 8. baseline_submit.csv 생성

`sample_submission.csv` 형식에 맞춰 최종 제출 파일을 생성합니다. 이 대회는 결과 CSV를 직접 제출하는
방식이므로, 이 파일을 그대로 제출하면 됩니다.


In [16]:
output_path = DATA_DIR / "baseline_submit.csv"
submission.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")


Saved: C:\Users\user\iCloudDrive\riversun\동국대학교\3학년 1학기\대외활동\DACON\데이터\open\baseline_submit.csv


In [17]:
import pandas as pd
import numpy as np

# 조사할 데이터셋 전체 등록
dfs_to_check = {
    "Train Labels": train_labels,
    "LDAPS Train": ldaps_train,
    "GFS Train": gfs_train,
    "SCADA Vestas": scada_vestas,
    "SCADA Unison": scada_unison
}

print("======================================================================")
print("🔍 [전수조사] 모든 데이터셋 컬럼별 이상치 & 노이즈 탐지 리포트")
print("======================================================================\n")

for name, df in dfs_to_check.items():
    print(f"📂 [데이터셋] {name} | 📐 크기: {df.shape}")
    print("-" * 70)
    
    # 수치형 컬럼만 추출
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) == 0:
        print("ℹ️ 수치형 컬럼이 존재하지 않습니다.\n")
        continue
        
    # 각 컬럼별 통계량 및 결측치 수집
    report_data = []
    for col in numeric_cols:
        # 결측치 계산
        null_cnt = df[col].isnull().sum()
        null_pct = (null_cnt / len(df)) * 100
        
        # 통계량 계산
        c_min = df[col].min()
        c_max = df[col].max()
        c_mean = df[col].mean()
        
        # 특이 변동성 (표준편차가 0이거나 미세한 경우 탐지)
        c_std = df[col].std()
        
        report_data.append({
            "Column Name": col,
            "Missing(Count)": null_cnt,
            "Missing(%)": f"{null_pct:.2f}%",
            "Min": round(c_min, 4),
            "Max": round(c_max, 4),
            "Mean": round(c_mean, 4),
            "Std": round(c_std, 4) if not np.isnan(c_std) else 0.0
        })
    
    # 데이터프레임으로 변환하여 깔끔하게 출력
    report_df = pd.DataFrame(report_data)
    
    # 🎯 이상치가 강력하게 의심되는 컬럼들을 가독성 좋게 보기 위해 판다스 출력 옵션 조정
    pd.set_option('display.max_rows', None)  # 모든 컬럼 다 찍기
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    
    print(report_df.to_string(index=False))
    print("======================================================================\n")

🔍 [전수조사] 모든 데이터셋 컬럼별 이상치 & 노이즈 탐지 리포트

📂 [데이터셋] Train Labels | 📐 크기: (26304, 4)
----------------------------------------------------------------------
Column Name  Missing(Count) Missing(%)  Min       Max      Mean       Std
kpx_group_1             104      0.40%  0.0 21275.305 6621.9811 6582.4431
kpx_group_2             103      0.39%  0.0 21362.084 7076.8429 7001.1461
kpx_group_3            8766     33.33%  0.0 21130.674 5563.8196 6294.5829

📂 [데이터셋] LDAPS Train | 📐 크기: (420864, 35)
----------------------------------------------------------------------
                 Column Name  Missing(Count) Missing(%)        Min         Max        Mean      Std
                     grid_id               0      0.00%     1.0000     16.0000      8.5000   4.6098
                    latitude               0      0.00%    37.2607     37.3032     37.2819   0.0138
                   longitude               0      0.00%   128.9257    128.9958    128.9607   0.0213
    heightAboveGround_10_10u           

In [18]:
# === [독립 실행용] 데이터 무결성 및 단위 검증 스캐너 ===
print("==================================================")
print("🔍 [기압 및 풍속 단위 검증] 데이터 샘플 스냅샷")
print("==================================================")

if 'ldaps_train' in globals() and not ldaps_train.empty:
    p_col_ldaps = [c for c in ldaps_train.columns if '_sp' in c or 'press' in c.lower()]
    if p_col_ldaps:
        print(f"▶ LDAPS 기압 원본 [{p_col_ldaps[0]}] 최소/최대값: {ldaps_train[p_col_ldaps[0]].min()} ~ {ldaps_train[p_col_ldaps[0]].max()}")
        print(f"   (값의 스케일이 100,000 안팎이면 Pa, 1,000 안팎이면 hPa입니다.)")

if 'gfs_train' in globals() and not gfs_train.empty:
    p_col_gfs = [c for c in gfs_train.columns if '_sp' in c or 'press' in c.lower()]
    if p_col_gfs:
        print(f"▶ GFS 기압 원본 [{p_col_gfs[0]}] 최소/최대값: {gfs_train[p_col_gfs[0]].min()} ~ {gfs_train[p_col_gfs[0]].max()}\n")

print("==================================================")
print("🔍 [결측치 및 미사용 컬럼 탐지] 요약 리포트")
print("==================================================")
for name, df in [("LDAPS Train", ldaps_train), ("GFS Train", gfs_train), ("Vestas SCADA", scada_vestas), ("Unison SCADA", scada_unison)]:
    null_counts = df.isnull().sum().sum()
    print(f"📂 {name} 데이터셋 전체 크기: {df.shape} | 총 결측치 수: {null_counts}")

🔍 [기압 및 풍속 단위 검증] 데이터 샘플 스냅샷
▶ LDAPS 기압 원본 [surface_0_sp] 최소/최대값: 87462.81 ~ 93083.13
   (값의 스케일이 100,000 안팎이면 Pa, 1,000 안팎이면 hPa입니다.)
▶ GFS 기압 원본 [surface_0_sp] 최소/최대값: 90569.836 ~ 103344.33

🔍 [결측치 및 미사용 컬럼 탐지] 요약 리포트
📂 LDAPS Train 데이터셋 전체 크기: (420864, 35) | 총 결측치 수: 0
📂 GFS Train 데이터셋 전체 크기: (236736, 40) | 총 결측치 수: 0
📂 Vestas SCADA 데이터셋 전체 크기: (157819, 41) | 총 결측치 수: 434
📂 Unison SCADA 데이터셋 전체 크기: (105264, 20) | 총 결측치 수: 9512


In [19]:
# === [독립 실행용] 원본 기상 변수 이름 전수 추출 ===
if 'ldaps_train' in globals() and 'train_set' in globals():
    # 현재 학습 피처에 포함된 컬럼 키워드 추출
    used_cols = list(X_train.columns)
    
    print("==================================================")
    print("📂 LDAPS 원본 데이터셋의 전체 컬럼 목록 (총 35개)")
    print("==================================================")
    print(list(ldaps_train.columns))
    
    print("\n==================================================")
    print("📂 GFS 원본 데이터셋의 전체 컬럼 목록 (총 40개)")
    print("==================================================")
    print(list(gfs_train.columns))

📂 LDAPS 원본 데이터셋의 전체 컬럼 목록 (총 35개)
['forecast_kst_dtm', 'data_available_kst_dtm', 'grid_id', 'latitude', 'longitude', 'heightAboveGround_10_10u', 'heightAboveGround_10_10v', 'heightAboveGround_50_50MUmax', 'heightAboveGround_50_50MUmin', 'heightAboveGround_50_50MVmax', 'heightAboveGround_50_50MVmin', 'heightAboveGround_5_XBLWS', 'heightAboveGround_5_YBLWS', 'heightAboveGround_2_t', 'heightAboveGround_2_dpt', 'heightAboveGround_2_r', 'heightAboveGround_2_q', 'surface_0_sp', 'meanSea_0_prmsl', 'etc_0_blh', 'surface_0_NDNSW', 'surface_0_NDNLW', 'heightAboveGround_2_SWDIR', 'heightAboveGround_2_SWDIF', 'etc_0_hcc', 'etc_0_mcc', 'etc_0_lcc', 'etc_0_VLCDC', 'surface_0_avg_lsprate', 'surface_0_lssrate', 'surface_0_ncpcp', 'surface_0_snol', 'surface_0_SNOM', 'surface_0_lsm', 'surface_0_h']

📂 GFS 원본 데이터셋의 전체 컬럼 목록 (총 40개)
['forecast_kst_dtm', 'data_available_kst_dtm', 'grid_id', 'latitude', 'longitude', 'heightAboveGround_10_10u', 'heightAboveGround_10_10v', 'heightAboveGround_80_u', 'heightAbo

In [20]:
# === [독립 실행용] 고고도 기상 변수 단위 및 스케일 검증 스캐너 ===
print("======================================================================")
print("🔍 [GFS 고고도 변수 검증] 지상 80m/100m 바람 및 상층 기온")
print("======================================================================\n")

if 'gfs_train' in globals() and not gfs_train.empty:
    gfs_check_cols = [
        'heightAboveGround_80_u', 'heightAboveGround_80_v',
        'heightAboveGround_100_100u', 'heightAboveGround_100_100v',
        'isobaricInhPa_850_t'
    ]
    
    for col in gfs_check_cols:
        if col in gfs_train.columns:
            c_min = gfs_train[col].min()
            c_max = gfs_train[col].max()
            c_mean = gfs_train[col].mean()
            c_null = gfs_train[col].isnull().sum()
            print(f"▶ 컬럼명: {col}")
            print(f"   - 범위  : {c_min:.4f} ~ {c_max:.4f} (평균: {c_mean:.4f})")
            print(f"   - 결측치: {c_null}개")
            if '_t' in col:
                print(f"   💡 [기온 판정힌트] 평균이 250~300 사이면 캘빈(K), -30~40 사이면 섭씨(℃)입니다.")
            else:
                print(f"   💡 [바람 판정힌트] 값이 -40~40 m/s 범위 내에 예쁘게 들어오는지 확인하세요.")
            print("-" * 60)

print("\n======================================================================")
print("🔍 [LDAPS 고고도 변수 검증] 지상 50m 바람 및 Wind Shear")
print("======================================================================\n")

if 'ldaps_train' in globals() and not ldaps_train.empty:
    ldaps_check_cols = [
        'heightAboveGround_50_50MUmax', 'heightAboveGround_50_50MUmin',
        'heightAboveGround_5_XBLWS', 'heightAboveGround_5_YBLWS'
    ]
    
    for col in ldaps_check_cols:
        if col in ldaps_train.columns:
            c_min = ldaps_train[col].min()
            c_max = ldaps_train[col].max()
            c_mean = ldaps_train[col].mean()
            c_null = ldaps_train[col].isnull().sum()
            print(f"▶ 컬럼명: {col}")
            print(f"   - 범위  : {c_min:.4f} ~ {c_max:.4f} (평균: {c_mean:.4f})")
            print(f"   - 결측치: {c_null}개")
            print("-" * 60)

🔍 [GFS 고고도 변수 검증] 지상 80m/100m 바람 및 상층 기온

▶ 컬럼명: heightAboveGround_80_u
   - 범위  : -21.8138 ~ 25.6597 (평균: 1.9280)
   - 결측치: 0개
   💡 [바람 판정힌트] 값이 -40~40 m/s 범위 내에 예쁘게 들어오는지 확인하세요.
------------------------------------------------------------
▶ 컬럼명: heightAboveGround_80_v
   - 범위  : -30.7273 ~ 19.6237 (평균: 0.2226)
   - 결측치: 0개
   💡 [바람 판정힌트] 값이 -40~40 m/s 범위 내에 예쁘게 들어오는지 확인하세요.
------------------------------------------------------------
▶ 컬럼명: heightAboveGround_100_100u
   - 범위  : -22.4202 ~ 27.1495 (평균: 2.0256)
   - 결측치: 0개
   💡 [바람 판정힌트] 값이 -40~40 m/s 범위 내에 예쁘게 들어오는지 확인하세요.
------------------------------------------------------------
▶ 컬럼명: heightAboveGround_100_100v
   - 범위  : -31.5098 ~ 20.1127 (평균: 0.2285)
   - 결측치: 0개
   💡 [바람 판정힌트] 값이 -40~40 m/s 범위 내에 예쁘게 들어오는지 확인하세요.
------------------------------------------------------------
▶ 컬럼명: isobaricInhPa_850_t
   - 범위  : 247.1055 ~ 297.8268 (평균: 279.1217)
   - 결측치: 0개
   💡 [기온 판정힌트] 평균이 250~300 사이면 캘빈(K), -30~40 사이면 섭씨(℃)입니다.
---------